# 06. Phase-invariant harmonic features and EVI2

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.pilots import climate, clustering, data, earth_engine, evaluation, features
from model_aza.training import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

## Feature sets

In [ ]:
optical = features.optical_indices(arrays["s2_values"])
names = features.OPTICAL_INDEX_NAMES
sets = {"harmonic_full_bands_indices": features.harmonic_features(cube, valid),
        "full_indices_evi2": features.harmonic_features(optical, valid, names=names, form="full"),
        "amplitude_indices_evi2": features.harmonic_features(optical, valid, names=names, form="amplitude"),
        "amplitude_relphase_indices_evi2": features.harmonic_features(optical, valid, names=names, form="invariant"),
        "amplitude_phasediff_indices_evi2": features.harmonic_features(optical, valid, names=names, form="phase_difference"),
        "amplitude_evi2_only": features.harmonic_features(optical[:, :, :1], valid, names=("EVI2",), form="amplitude")}
common = np.logical_and.reduce([ok for _, ok, _ in sets.values()])
pd.DataFrame([{"feature_set": k, "predictors": X.shape[1], "pixels": int(ok.sum())} for k, (X, ok, _) in sets.items()])

## Evaluate on pixels usable by every set

In [ ]:
run = tuning.new_run(root, "phase_invariant", subset_run / "meta.json", area=AREA)
summaries, details = [], []
for name, (X, _, _) in sets.items():
    s, d = repeated(name, np.nan_to_num(X), rows, common)
    summaries += s
    details += d
    print(name, "done", flush=True)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "dataset_holdout", details)
display(summarize(summaries))

## Figure

In [ ]:
table = pd.DataFrame(summaries).groupby("feature_set", sort=False)[
    ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1"]].mean()
figure, axis = plt.subplots(figsize=(12, 4))
table.T.plot(kind="bar", ax=axis)
axis.set_ylim(0, 1)
axis.set_ylabel("Parcel macro-F1 (mean of seeds)")
axis.legend(fontsize=7)
show_and_save(figure, run, "phase_invariant")
print("Run folder:", run)